In [ ]:
import numpy as np
from numpy.random import default_rng

import matplotlib.pyplot as plt
from util.Processing import discretize, td_nnlsr_deconvolve
from util.Plotting import plot_photons

from scipy import signal

In [ ]:
n = 10
bits = 8
gap = 5
mag = 100
sampling_ratio = 1
ref_mag = 1000

In [ ]:
def nai_pulse(a, n=-1, sampling_ratio=1):
    sos1 = signal.butter(3, 0.25e7, btype='low', analog=False, output='sos', fs=40e6*sampling_ratio)
    sos2 = signal.butter(2, .156e7, btype='low', analog=False, output='sos', fs=40e6*sampling_ratio)
    sos3 = signal.butter(1, .053e7, btype='low', analog=False, output='sos', fs=40e6*sampling_ratio)

    start=0
    stop = 2.5e-6
    dt = 25e-9 / sampling_ratio
    t = np.arange(start, stop, dt)[:-1]
    
    y = t * 0
    y[4*sampling_ratio] = 1
    fil1 = signal.sosfilt(sos1, y)
    fil1 = fil1*a/np.max(fil1)
    fil2 = signal.sosfilt(sos2, y)
    fil2 = fil2*a/np.max(fil2)
    fil3 = signal.sosfilt(sos3, y)
    fil3 = fil3*a/np.max(fil3)
    fil = np.concatenate((fil1[0*sampling_ratio:11*sampling_ratio],
                          fil2[11*sampling_ratio:17*sampling_ratio],
                          fil3[17*sampling_ratio:]))
    return t, fil

In [ ]:
plt.figure(figsize=(10,4), dpi=200)

t, kernel = nai_pulse(1, sampling_ratio=10)
plt.plot(t, kernel, 'b.')
print(kernel.size)
t, kernel = nai_pulse(1, sampling_ratio=1)
plt.plot(t, kernel, 'r.')
print(kernel.size)

plt.xlim([.2E-6, .5E-6])

In [ ]:

trace = np.zeros((n+2) * kernel.size) #rough in-exact preallocation to trim latter
volts_list = []
index_list = []

# Build directly so we can get good spacing between clusters for independence
start = 0
for spacing in range(1, n+1):
    
    trace[start: start+kernel.size] += mag * kernel
    volts_list.append(mag)
    index_list.append(start)
    start+=spacing
    
    trace[start: start+kernel.size] += mag * kernel
    volts_list.append(mag)
    index_list.append(start)
    start+=kernel.size + gap
    
if start < trace.size: # trim a bit
    trace = trace[:start]
    
trace = discretize(trace, bits=bits)
energy_list = np.array(volts_list)
index_list = np.array(index_list)

print(energy_list)
print(index_list)

    
fig, axes = plt.subplots(3, 1, figsize=(8, 6), dpi=200)
plot_photons(structure=axes[0], photons_=index_list, magnitude=energy_list, color='red',label_='Photons', alpha=1)
axes[0].legend()
axes[0].set_xlim(-10, trace.size)
axes[1].plot(trace, label='Trace')
axes[1].legend()
axes[1].set_xlim(-10, trace.size)


discretized_kernel = discretize(ref_mag*nai_pulse(1)[1], bits=bits)/ref_mag
deconv = td_nnlsr_deconvolve(trace, discretized_kernel)

gt0 = deconv > 0
ngt0 = deconv <= 0
axes[2].plot(np.arange(deconv.size)[gt0], deconv[gt0], marker='.', linestyle='', color='cyan', label='Deconvolution > 0')
axes[2].plot(np.arange(deconv.size)[ngt0], deconv[ngt0], marker='.', linestyle='', color='cyan', alpha=.1, label='Deconvolution = 0')
axes[2].set_xlim(-10, trace.size)


In [ ]:
# TODO note: Could try assuming that small magnitude and larger magnitude points near each other should be summed...